In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import requests
import time

engine = create_engine('sqlite:///terraria.db')
df = pd.read_excel('terraria-armors.xlsx')

df.head()

,Name,Category,Stage,Image
0,Mining Helmet,1,NaN,NaN
1,Mining Shirt,2,NaN,NaN
2,Mining Pants,3,NaN,NaN
3,Wood Helmet,1,NaN,NaN
4,Wood Breastplate,2,NaN,NaN


In [33]:
df.loc[df["Name"] == "Flix Fur Coat", "Name"] = "Flinx Fur Coat"

In [34]:
BASE_URL = 'https://terraria.wiki.gg/images/'

# Create the column if needed, preserving existing results
if 'Image' not in df.columns:
    df['Image'] = pd.Series(index=df.index, dtype='object')
else:
    df['Image'] = df['Image'].astype('object')

for index, row in df.iterrows():

    # Skip armor already checked successfully
    if pd.notna(row['Image']):
        continue

    armor = row['Name'].replace(" ", "_").replace("'","%27")
    

    url = BASE_URL + armor + '.png'
    response = requests.get(url, timeout=15)
    print(url)
    print(response.status_code)

    # Respect rate limits and stop safely
    if response.status_code == 429:
        wait = response.headers.get('Retry-After', '60')
        print(f"Rate limited. Retry after {wait} seconds.")
        print(f"Stopped at row {index}. Rerun after the cooldown.")
        break

    if response.status_code == 200:
        df.at[index, 'Image'] = url

    elif response.status_code == 404:
        url = BASE_URL + armor + '.gif'
        response = requests.get(url, timeout=15)

        if response.status_code == 429:
            wait = response.headers.get('Retry-After', '60')
            print(f"Rate limited. Retry after {wait} seconds.")
            print(f"Stopped at row {index}. Rerun after the cooldown.")
            break

        if response.status_code == 200:
            df.at[index, 'Image'] = url

    # Pause between armor
    time.sleep(2)

print("Image lookup pass finished.")
print("Images found:", df['Image'].notna().sum())
print("Images remaining:", df['Image'].isna().sum())

https://terraria.wiki.gg/images/Flinx_Fur_Coat.png
200
https://terraria.wiki.gg/images/Gi.png
200
https://terraria.wiki.gg/images/Djinn%27s_Curse.png
200
https://terraria.wiki.gg/images/Moon_Lord_Legs.png
200
https://terraria.wiki.gg/images/Dead_Man%27s_Sweater.png
200
Image lookup pass finished.
Images found: 256
Images remaining: 0


In [35]:
df.to_sql(
    name='armor',
    con=engine,
    if_exists='replace',
    index=False
)

256